# ch04 Bonus 02：分组查询注意力（GQA）

> 对照官方 `rasbt/LLMs-from-scratch` ch04/04_gqa
> **参考真实模型**：Llama-2 / Llama-3 / Qwen

## 一句话

让多个 Query 头**共享**同一组 K/V，在'质量≈MHA'和'省显存≈MQA'之间取折中。KV 缓存显存直接按 `num_kv_groups` 缩减。

## 为什么需要 GQA

推理时 KV 缓存占的显存随层数 × 头数线性增长，是长上下文的显存瓶颈：

| 方案 | K/V 头数 | KV 缓存显存 | 质量 |
|------|---------|------------|------|
| MHA（标准多头） | = Q 头数 | 最大 | 最好 |
| MQA（多查询）   | = 1     | 最小（1/n） | 略降 |
| **GQA**         | 分组数（介于两者） | 中间 | **≈MHA** |

> GQA = MHA 和 MQA 的连续插值。`num_kv_groups = num_heads` 时退化为 MHA；`num_kv_groups = 1` 时为 MQA。

## 核心改造

标准 MHA 的 `W_query / W_key / W_value` 都投影到 `d_out = num_heads × head_dim`。GQA 唯一区别：**K/V 的投影维度缩小到 `num_kv_groups × head_dim`**，前向时用 `repeat_interleave` 把每组 K/V 复制到对应的 Q 头数。

## 1. 实现 GQA

和 `src/gpt/attention.py` 的 `MultiHeadAttention` 对照，唯一改动是 `W_key/W_value` 的输出维度和 `repeat_interleave` 这一步。

In [ ]:
import torch
import torch.nn as nn


class GroupedQueryAttention(nn.Module):
    """分组查询注意力（GQA）。num_kv_groups=1 即 MQA，=num_heads 即 MHA。"""

    def __init__(self, d_in, d_out, context_length, num_heads, num_kv_groups,
                 dropout=0.0, qkv_bias=False):
        super().__init__();
        assert d_out % num_heads == 0, "d_out 必须能被 num_heads 整除"
        assert num_heads % num_kv_groups == 0, "num_heads 必须能被 num_kv_groups 整除"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups  # 每个 kv 组被几个 Q 头共享

        # 关键区别：K/V 只投影到 num_kv_groups 个组，而不是全部 num_heads 个头
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool(),
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys   = self.W_key(x).view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 把每组 K/V 复制 group_size 份，凑齐 num_heads 个头
        keys   = keys.repeat_interleave(self.group_size, dim=1)   # [b, num_heads, seq, head_dim]
        values = values.repeat_interleave(self.group_size, dim=1)

        attn_scores = queries @ keys.transpose(2, 3)              # [b, heads, seq, seq]
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

## 2. 三种方案对比：MHA / GQA / MQA

同样 `num_heads=8`、`d_out=768`，只改 `num_kv_groups`：

In [ ]:
torch.manual_seed(123)
batch, seq, dim = 2, 16, 768
x = torch.randn(batch, seq, dim)

print(f"{'方案':<6} {'KV组数':<8} {'输出形状':<20} {'参数量':<12}")
print("-" * 50)
for name, kv_groups in [("MHA", 8), ("GQA", 2), ("MQA", 1)]:
    layer = GroupedQueryAttention(dim, dim, 1024, num_heads=8, num_kv_groups=kv_groups)
    out = layer(x)
    params = sum(p.numel() for p in layer.parameters())
    print(f"{name:<6} {kv_groups:<8} {str(tuple(out.shape)):<20} {params:,}")

print("\n💡 KV 缓存显存同样按 num_kv_groups 比例缩减——这正是 GQA 推理省显存的关键。")

## 3. 放进 GPT

把 `TransformerBlock` 里的 `MultiHeadAttention` 换成 `GroupedQueryAttention` 即可，其余结构不变。Llama-3-8B 用 `num_kv_groups=2`（8 个 Q 头共享 2 组 KV），显著降低了长上下文推理的显存。

---
> 📌 本 notebook 实现 GQA 的核心类并验证 MHA/GQA/MQA 三档对比。
> 完整集成进 GPT 的实现见官方 `ch04/04_gqa/gqa.py`。